# LB-0015 — R2 legend vs. external-public-math continuation

Creates two 831-row public-leaderboard CSVs with the repository's full frozen SC16/PAL4/adaptive-length pipeline. Both adapters receive the same 2,048 → 4,096 → 8,192 conditional length route, so their leaderboard difference measures the model change under final deployment conditions.

In [ ]:
# Cell 1 — Use a fresh A100 runtime. Restart once after installation, then run Cells 2–4.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart runtime once now.")


In [ ]:
# Cell 2 — Clone/update the repo and cache the exact Qwen base. GITHUB_TOKEN is needed only while the repo is private.
from google.colab import drive, userdata
from pathlib import Path
import subprocess
drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = "https://github.com/jhparktime/qwen-math-final-2026.git"
if token:
    url = "https://x-access-token:" + token + "@github.com/jhparktime/qwen-math-final-2026.git"
repo = Path("/content/qwen-math-final")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Build immutable run configs: same SC16/PAL, no adaptive-length answer replacement.
import hashlib, json, re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r"[\s_-]+", "", unicodedata.normalize("NFC", str(value)).casefold())
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

roots = [p for p in Path("/content/drive/MyDrive").iterdir() if p.is_dir() and compact(p.name) == compact("2026소중한챌린지")]
assert len(roots) == 1
project = roots[0]
runs = project / "runs"
leaderboard = project / "data" / "deep_chal_math_leaderboard_filtered.csv"
assert leaderboard.exists()
parent_adapter = runs / "RFT-0004B-r2-pro4-hint-lowdrift-lora" / "adapter_final"
external_adapter = runs / "RFT-0015-r2pro4-external14k-lowdrift-r16" / "adapter_final"
for path in [parent_adapter, external_adapter]:
    assert (path / "adapter_config.json").exists() and (path / "adapter_model.safetensors").exists(), path

base_config = json.loads((Path("configs") / "final_inference.json").read_text())
configs = {}
for label, adapter in {"r2_legend": parent_adapter, "external14k": external_adapter}.items():
    config = json.loads(json.dumps(base_config))
    config["run_id"] = f"LB-0015-{label}-sc16-pal3"
    config["expected_rows"] = 831
    config["model"]["adapter_name"] = label
    config["model"]["adapter_weight_sha256"] = sha256_file(adapter / "adapter_model.safetensors")
    config["adaptive_length"]["router"]["top_gain_min"] = 999
    config["adaptive_length"]["router"]["margin_gain_min"] = 999
    config["adaptive_length"]["router"]["extended_top_min"] = 999
    config["adaptive_length"]["router"]["status"] = "disabled for exact R2+PAL score reproduction"
    config_path = Path("/content") / f"{label}_sc16_pal3_config.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    configs[label] = {"adapter": adapter, "config": config_path, "output": runs / f"LB-0015-{label}-sc16-pal3"}
print(json.dumps({key:{"adapter":str(value["adapter"]),"sha256":sha256_file(value["adapter"] / "adapter_model.safetensors"),"output":str(value["output"])} for key,value in configs.items()}, indent=2))


In [ ]:
# Cell 4 — Generate both full-pipeline SC16+PAL3+adaptive submissions. Resume-safe; do not run concurrently with dev evaluation.
for label, item in configs.items():
    config = json.loads(item["config"].read_text())
    config["adaptive_length"]["router"] = base_config["adaptive_length"]["router"]
    config["adaptive_length"]["router"]["status"] = "full final deployment route enabled for paired leaderboard comparison"
    item["config"].write_text(json.dumps(config, indent=2), encoding="utf-8")
    print(f"[RUN] {label}")
    !PYTHONPATH=. python3 inference/final_inference.py --input {leaderboard} --adapter {item['adapter']} --output-dir {item['output']} --config {item['config']}
    !python3 scripts/validate_submission.py --input {leaderboard} --submission {item['output'] / 'submissions/submission.csv'} --expected-rows 831
    target = Path("/content/drive/MyDrive") / f"submission_{label}_sc16_pal3_adaptive.csv"
    target.write_bytes((item['output'] / "submissions/submission.csv").read_bytes())
    print("[SUBMIT]", target)


In [ ]:
# Final cell — optional GPU release after both files are copied.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True after completion.")
